In [0]:
# abstract class to result data frames
class DataSink:
    def __init__(self,df,path,method,params):
        self.path = path
        self.method = method
        self.df = df
        self.params = params 

    # abstract method to read data sources
    def load_data_frame(self):    
        raise valueError("Subclass must implement abstract method")

# to write resulted data to DBFS
class LoadToDBFS(DataSink):
    
    def load_data_frame(self):
        self.df.write.format("parquet").mode(self.method).save(self.path)
        

# to write resulted data to DBFS with partition
class LoadToDBFSWithPartition(DataSink):

    def load_data_frame(self):        
        partitionByColumns = self.params.get("partitionByColumns")
        self.df.write.format("parquet").mode(self.method).partitionBy(*partitionByColumns).save(self.path) 


# Write data to delta table
class LoadToDeltaTable(DataSink):

    def load_data_frame(self):
        self.df.write.format("delta").mode(self.method).save(self.path)
        #self.df.write.format("delta").mode(self.method).saveAsTable(self.path)

def get_sink_source(sink_type, df, path, method, params = None):
    print(f"sink type : {sink_type}")

    if sink_type == "dbfs":
        return LoadToDBFS(df, path, method, params)
    elif sink_type == "dbfs_with_partition":
        return LoadToDBFSWithPartition(df, path, method, params)
    elif sink_type == "delta":
        return LoadToDeltaTable(df, path, method, params)
    else:
        return ValueError(f"Not implemented for sink type: {sink_type}")